# 🧠 Transfer Learning Showdown: AlexNet vs VGG16 vs ResNet50 vs EfficientNetB0

### Image Classification on CIFAR-10 using Pre-trained CNNs

---

**What this notebook does**
1. Loads a balanced subset of **CIFAR-10** for fast, GPU-friendly experimentation
2. Applies **transfer learning** (frozen convolutional base + retrained classifier head) to four famous architectures
3. Trains, evaluates, and **compares** all four models side-by-side
4. Produces rich visualizations: training curves, confusion matrices, metric bar charts, a radar chart, model-efficiency scatter plots, and sample predictions
5. Ends with an auto-generated **leaderboard** ranking the models



## Table of Contents
1. [Setup & Imports](#setup)
2. [Dataset Preparation](#dataset)
3. [Sample Data Visualization](#sample-viz)
4. [Model Building (Transfer Learning)](#build)
5. [Training & Evaluation Functions](#train-eval)
6. [Run All Four Models](#run)
7. [Results & Visual Comparison](#results)
8. [Confusion Matrices](#confusion)
9. [Model Efficiency Analysis](#efficiency)
10. [Radar Chart — Multi-metric View](#radar)
11. [Sample Predictions](#predictions)
12. [Final Leaderboard & Conclusion](#conclusion)


## 1. Setup & Imports <a id='setup'></a>

In [1]:
# Uncomment if a package is missing in your environment
# !pip install torch torchvision scikit-learn seaborn matplotlib pandas --quiet

import os, time, copy, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset

import torchvision
from torchvision import datasets, transforms, models

from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                              confusion_matrix, classification_report)

import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Using device: {device}")
if device.type == 'cuda':
    print(f"🖥️  GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  No GPU detected — training will be slower. Enable a GPU runtime for best results.")


✅ Using device: cuda
🖥️  GPU: Tesla T4


## 2. Dataset Preparation <a id='dataset'></a>

We use **CIFAR-10** (10 balanced classes, 32×32 native resolution). Since all four
pre-trained networks expect ImageNet-style 224×224 RGB inputs, we resize and
normalize using ImageNet statistics. To keep training fast on Colab, we sample a
**stratified subset** rather than the full 50,000/10,000 split.

In [ ]:
CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck']
NUM_CLASSES = len(CLASSES)

IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

full_train      = datasets.CIFAR10(root='./data', train=True,  download=True, transform=train_transform)
full_train_eval = datasets.CIFAR10(root='./data', train=True,  download=True, transform=eval_transform)
full_test       = datasets.CIFAR10(root='./data', train=False, download=True, transform=eval_transform)

def stratified_indices(dataset, per_class, seed=SEED, exclude=None):
    rng = np.random.RandomState(seed)
    targets = np.array(dataset.targets)
    exclude = set(exclude) if exclude else set()
    idxs = []
    for c in range(NUM_CLASSES):
        class_idx = np.array([i for i in np.where(targets == c)[0] if i not in exclude])
        rng.shuffle(class_idx)
        idxs.extend(class_idx[:per_class])
    rng.shuffle(idxs)
    return idxs

train_idx = stratified_indices(full_train, per_class=400, seed=SEED)
val_idx   = stratified_indices(full_train, per_class=80,  seed=SEED + 1, exclude=train_idx)
test_idx  = stratified_indices(full_test,  per_class=100, seed=SEED + 2)

train_ds = Subset(full_train, train_idx)
val_ds   = Subset(full_train_eval, val_idx)
test_ds  = Subset(full_test, test_idx)

BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"📦 Train: {len(train_ds)} images | Val: {len(val_ds)} images | Test: {len(test_ds)} images")
print(f"📚 Classes ({NUM_CLASSES}): {CLASSES}")


 21%|██        | 35.6M/170M [09:21<34:30, 65.2kB/s]

## 3. Sample Data Visualization <a id='sample-viz'></a>

In [ ]:
def show_sample_grid(n=10, title="Sample Images from CIFAR-10"):
    raw_view = datasets.CIFAR10(root='./data', train=True, download=True)
    fig, axes = plt.subplots(2, 5, figsize=(14, 6))
    fig.suptitle(title, fontsize=16, fontweight='bold')
    idxs = random.sample(range(len(raw_view)), n)
    for ax, i in zip(axes.flat, idxs):
        img, label = raw_view[i]
        ax.imshow(img)
        ax.set_title(CLASSES[label], fontsize=11)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

show_sample_grid()

# Class distribution in our training subset
fig, ax = plt.subplots(figsize=(10, 4))
targets = np.array(full_train.targets)[train_idx]
counts = [np.sum(targets == c) for c in range(NUM_CLASSES)]
bars = ax.bar(CLASSES, counts, color=sns.color_palette('husl', NUM_CLASSES))
ax.set_title('Training Subset — Class Balance', fontsize=14, fontweight='bold')
ax.set_ylabel('# Images')
plt.xticks(rotation=30)
for b, c in zip(bars, counts):
    ax.text(b.get_x() + b.get_width()/2, c + 3, str(c), ha='center', fontsize=9)
plt.tight_layout()
plt.show()


## 4. Model Building — Transfer Learning <a id='build'></a>

For each architecture we:
- Load **ImageNet pre-trained weights**
- **Freeze** the convolutional feature-extraction layers (they already know general
  visual features like edges, textures, and shapes)
- **Replace** the final classification layer with a new fully-connected layer
  matching our 10 CIFAR-10 classes
- Train **only** the new head (fast, low risk of overfitting on a small dataset)

In [ ]:
def build_model(name, num_classes=NUM_CLASSES, freeze_base=True):
    '''Builds a transfer-learning model for the given architecture name.'''
    if name == 'AlexNet':
        model = models.alexnet(weights=models.AlexNet_Weights.IMAGENET1K_V1)
        if freeze_base:
            for p in model.features.parameters():
                p.requires_grad = False
        in_feats = model.classifier[6].in_features
        model.classifier[6] = nn.Linear(in_feats, num_classes)

    elif name == 'VGG16':
        model = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
        if freeze_base:
            for p in model.features.parameters():
                p.requires_grad = False
        in_feats = model.classifier[6].in_features
        model.classifier[6] = nn.Linear(in_feats, num_classes)

    elif name == 'ResNet50':
        model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        if freeze_base:
            for p in model.parameters():
                p.requires_grad = False
        in_feats = model.fc.in_features
        model.fc = nn.Linear(in_feats, num_classes)

    elif name == 'EfficientNetB0':
        model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        if freeze_base:
            for p in model.features.parameters():
                p.requires_grad = False
        in_feats = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(in_feats, num_classes)

    else:
        raise ValueError(f"Unknown model name: {name}")

    return model.to(device)


def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


## 5. Training & Evaluation Functions <a id='train-eval'></a>

In [ ]:
def train_model(name, model, train_loader, val_loader, epochs=6, lr=1e-3):
    criterion = nn.CrossEntropyLoss()
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.Adam(trainable_params, lr=lr)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)

    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_val_acc = 0.0
    best_state = copy.deepcopy(model.state_dict())

    start = time.time()
    for epoch in range(epochs):
        model.train()
        running_loss, running_correct, total = 0.0, 0, 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * imgs.size(0)
            running_correct += (outputs.argmax(1) == labels).sum().item()
            total += imgs.size(0)

        train_loss = running_loss / total
        train_acc = running_correct / total

        model.eval()
        v_loss, v_correct, v_total = 0.0, 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                outputs = model(imgs)
                loss = criterion(outputs, labels)
                v_loss += loss.item() * imgs.size(0)
                v_correct += (outputs.argmax(1) == labels).sum().item()
                v_total += imgs.size(0)
        val_loss = v_loss / v_total
        val_acc = v_correct / v_total
        scheduler.step()

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        print(f"[{name}] Epoch {epoch+1}/{epochs} | "
              f"train_loss={train_loss:.4f} acc={train_acc:.4f} | "
              f"val_loss={val_loss:.4f} acc={val_acc:.4f}")

    total_time = time.time() - start
    model.load_state_dict(best_state)
    print(f"✅ [{name}] Done in {total_time:.1f}s | Best val acc: {best_val_acc:.4f}\n")
    return model, history, total_time


def evaluate_model(name, model, test_loader):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    start = time.time()
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs = imgs.to(device)
            outputs = model(imgs)
            probs = torch.softmax(outputs, dim=1)
            preds = outputs.argmax(1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
            all_probs.extend(probs.cpu().numpy())
    inference_time = (time.time() - start) / len(test_loader.dataset) * 1000  # ms/image

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs  = np.array(all_probs)

    acc = accuracy_score(all_labels, all_preds)
    prec, rec, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average='weighted', zero_division=0)
    cm = confusion_matrix(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, target_names=CLASSES, zero_division=0)

    return {
        'name': name, 'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1,
        'cm': cm, 'report': report, 'preds': all_preds, 'labels': all_labels,
        'probs': all_probs, 'inference_ms': inference_time
    }


## 6. Run All Four Models <a id='run'></a>

This cell trains AlexNet, VGG16, ResNet50, and EfficientNetB0 back-to-back using
identical settings for a fair comparison. On a Colab GPU this typically takes a
few minutes total.

In [ ]:
MODEL_NAMES = ['AlexNet', 'VGG16', 'ResNet50', 'EfficientNetB0']
EPOCHS = 6

results       = {}
histories     = {}
param_counts  = {}
train_times   = {}

for name in MODEL_NAMES:
    print("=" * 72)
    print(f" 🚀 TRANSFER LEARNING: {name}")
    print("=" * 72)

    model = build_model(name, freeze_base=True)
    total_p, trainable_p = count_params(model)
    param_counts[name] = (total_p, trainable_p)
    print(f"Total params: {total_p:,} | Trainable (head only): {trainable_p:,}")

    model, history, t_time = train_model(name, model, train_loader, val_loader, epochs=EPOCHS)
    histories[name] = history
    train_times[name] = t_time

    eval_result = evaluate_model(name, model, test_loader)
    results[name] = eval_result
    print(f"📊 [{name}] TEST -> acc={eval_result['accuracy']:.4f} "
          f"prec={eval_result['precision']:.4f} rec={eval_result['recall']:.4f} "
          f"f1={eval_result['f1']:.4f}\n")

    del model
    if device.type == 'cuda':
        torch.cuda.empty_cache()

print("🎉 All models trained and evaluated!")


## 7. Results & Visual Comparison <a id='results'></a>

In [ ]:
colors = {'AlexNet': '#e74c3c', 'VGG16': '#3498db', 'ResNet50': '#2ecc71', 'EfficientNetB0': '#9b59b6'}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for name in MODEL_NAMES:
    axes[0].plot(range(1, EPOCHS + 1), histories[name]['val_acc'], marker='o',
                 label=name, color=colors[name], linewidth=2)
    axes[1].plot(range(1, EPOCHS + 1), histories[name]['val_loss'], marker='o',
                 label=name, color=colors[name], linewidth=2)

axes[0].set_title('Validation Accuracy per Epoch', fontweight='bold', fontsize=13)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].set_title('Validation Loss per Epoch', fontweight='bold', fontsize=13)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('validation_curves.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, name in zip(axes.flat, MODEL_NAMES):
    h = histories[name]
    ax.plot(h['train_acc'], label='Train Acc', linestyle='--', color=colors[name])
    ax.plot(h['val_acc'],   label='Val Acc',   color=colors[name], linewidth=2)
    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
    ax.legend(); ax.grid(alpha=0.3)

plt.suptitle('Train vs Validation Accuracy — All Models', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('train_vs_val_all_models.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Build a tidy summary dataframe
summary_data = []
for name in MODEL_NAMES:
    r = results[name]
    total_p, trainable_p = param_counts[name]
    summary_data.append({
        'Model': name,
        'Test Accuracy': r['accuracy'],
        'Precision': r['precision'],
        'Recall': r['recall'],
        'F1-Score': r['f1'],
        'Train Time (s)': train_times[name],
        'Inference (ms/img)': r['inference_ms'],
        'Total Params (M)': total_p / 1e6,
        'Trainable Params (M)': trainable_p / 1e6
    })

summary_df = pd.DataFrame(summary_data).sort_values('Test Accuracy', ascending=False).reset_index(drop=True)

summary_df_display = summary_df.copy()
for col in ['Test Accuracy', 'Precision', 'Recall', 'F1-Score']:
    summary_df_display[col] = summary_df_display[col].round(4)
for col in ['Train Time (s)', 'Inference (ms/img)', 'Total Params (M)', 'Trainable Params (M)']:
    summary_df_display[col] = summary_df_display[col].round(2)

styled = (summary_df_display.style
          .background_gradient(subset=['Test Accuracy', 'F1-Score'], cmap='Greens')
          .background_gradient(subset=['Train Time (s)', 'Inference (ms/img)'], cmap='Reds_r')
          .set_caption('📊 Transfer Learning Model Comparison Summary')
          .set_table_styles([{'selector': 'caption',
                               'props': [('font-size', '16px'), ('font-weight', 'bold')]}]))
styled


In [ ]:
metrics = ['accuracy', 'precision', 'recall', 'f1']
x = np.arange(len(MODEL_NAMES))
width = 0.2

fig, ax = plt.subplots(figsize=(12, 6))
for i, m in enumerate(metrics):
    vals = [results[n][m] for n in MODEL_NAMES]
    bars = ax.bar(x + i * width, vals, width, label=m.capitalize())

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(MODEL_NAMES)
ax.set_ylabel('Score')
ax.set_ylim(0, 1)
ax.set_title('Performance Metrics Comparison', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


## 8. Confusion Matrices <a id='confusion'></a>

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
for ax, name in zip(axes.flat, MODEL_NAMES):
    cm = results[name]['cm']
    cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=CLASSES, yticklabels=CLASSES, ax=ax, cbar=False)
    ax.set_title(f"{name} (Acc: {results[name]['accuracy']:.3f})", fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('Normalized Confusion Matrices', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Full per-class classification reports (precision/recall/F1 per class)
for name in MODEL_NAMES:
    print("=" * 60)
    print(f" Classification Report — {name}")
    print("=" * 60)
    print(results[name]['report'])


## 9. Model Efficiency Analysis <a id='efficiency'></a>

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for name in MODEL_NAMES:
    total_p, _ = param_counts[name]
    acc_pct = results[name]['accuracy'] * 100

    axes[0].scatter(total_p / 1e6, acc_pct, s=200, color=colors[name],
                     label=name, edgecolors='black', linewidth=1.2)
    axes[0].annotate(name, (total_p / 1e6, acc_pct),
                      textcoords="offset points", xytext=(8, 5), fontsize=9)

    axes[1].scatter(train_times[name], acc_pct, s=200, color=colors[name],
                     label=name, edgecolors='black', linewidth=1.2)
    axes[1].annotate(name, (train_times[name], acc_pct),
                      textcoords="offset points", xytext=(8, 5), fontsize=9)

axes[0].set_xlabel('Total Parameters (Millions)')
axes[0].set_ylabel('Test Accuracy (%)')
axes[0].set_title('Model Size vs Accuracy', fontweight='bold')
axes[0].grid(alpha=0.3)

axes[1].set_xlabel('Training Time (s)')
axes[1].set_ylabel('Test Accuracy (%)')
axes[1].set_title('Training Time vs Accuracy', fontweight='bold')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('efficiency_scatter.png', dpi=150, bbox_inches='tight')
plt.show()


## 10. Radar Chart — Multi-Metric View <a id='radar'></a>

In [ ]:
from math import pi

categories = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'Speed*']
max_inf = max(results[n]['inference_ms'] for n in MODEL_NAMES)

def get_stats(name):
    r = results[name]
    speed_score = 1 - (r['inference_ms'] / max_inf)  # higher = faster
    return [r['accuracy'], r['precision'], r['recall'], r['f1'], speed_score]

N = len(categories)
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))
for name in MODEL_NAMES:
    stats = get_stats(name)
    stats += stats[:1]
    ax.plot(angles, stats, linewidth=2, label=name, color=colors[name])
    ax.fill(angles, stats, alpha=0.08, color=colors[name])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=11)
ax.set_yticklabels([])
ax.set_title('Multi-Metric Model Comparison (Radar)\n*Speed is relative — higher is faster',
             fontsize=14, fontweight='bold', y=1.1)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1))
plt.tight_layout()
plt.savefig('radar_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


## 11. Sample Predictions <a id='predictions'></a>

In [ ]:
best_name = summary_df.iloc[0]['Model']
print(f"🏆 Best performing model on test set: {best_name}")

best_result = results[best_name]
correct_idx = np.where(best_result['preds'] == best_result['labels'])[0]
wrong_idx   = np.where(best_result['preds'] != best_result['labels'])[0]

raw_test = datasets.CIFAR10(root='./data', train=False, download=True)

fig, axes = plt.subplots(2, 6, figsize=(18, 7))

sample_correct = random.sample(list(correct_idx), min(6, len(correct_idx)))
sample_wrong   = random.sample(list(wrong_idx), min(6, len(wrong_idx))) if len(wrong_idx) else []

for ax, i in zip(axes[0], sample_correct):
    real_idx = test_idx[i]
    img, _ = raw_test[real_idx]
    ax.imshow(img)
    ax.set_title(f"✓ {CLASSES[best_result['labels'][i]]}", color='green', fontsize=10)
    ax.axis('off')

for ax in axes[0][len(sample_correct):]:
    ax.axis('off')

for ax, i in zip(axes[1], sample_wrong):
    real_idx = test_idx[i]
    img, _ = raw_test[real_idx]
    ax.imshow(img)
    ax.set_title(f"✗ True: {CLASSES[best_result['labels'][i]]}\nPred: {CLASSES[best_result['preds'][i]]}",
                 color='red', fontsize=9)
    ax.axis('off')

for ax in axes[1][len(sample_wrong):]:
    ax.axis('off')

fig.suptitle(f'{best_name} — Correct (top row) vs Incorrect (bottom row) Predictions',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('sample_predictions.png', dpi=150, bbox_inches='tight')
plt.show()


## 12. Final Leaderboard & Conclusion <a id='conclusion'></a>

In [ ]:
print("=" * 72)
print(" 🏁 FINAL LEADERBOARD")
print("=" * 72)
print(summary_df_display.to_string(index=False))

print("\n🏆 Best Test Accuracy   :", summary_df.iloc[0]['Model'],
      f"({summary_df.iloc[0]['Test Accuracy']:.2%})")
print("⚡ Fastest Inference    :", summary_df.sort_values('Inference (ms/img)').iloc[0]['Model'])
print("🪶 Smallest Model       :", summary_df.sort_values('Total Params (M)').iloc[0]['Model'])
print("⏱️  Fastest to Train     :", summary_df.sort_values('Train Time (s)').iloc[0]['Model'])


### 📝 Discussion

- **Accuracy vs. Model Size**: Deeper/larger models (VGG16, ResNet50) don't always
  win — EfficientNetB0 is designed via compound scaling to balance depth, width,
  and resolution, often matching larger models with far fewer parameters.
- **AlexNet as a baseline**: Being the oldest and shallowest architecture here,
  AlexNet typically trails the others, illustrating over a decade of architectural
  progress (skip connections in ResNet, deeper stacks in VGG, and efficient
  scaling in EfficientNet).
- **Frozen-base transfer learning** trains only the classifier head, which is why
  training finishes in minutes — the convolutional layers already encode rich,
  transferable ImageNet features.
- **Trade-offs matter in practice**: the "best" model depends on the constraint —
  a mobile app cares about inference time and parameter count, while a research
  benchmark cares purely about accuracy.

### 🔧 Ideas to Extend This Notebook
- Unfreeze the last few convolutional blocks and **fine-tune** with a low learning
  rate for a further accuracy boost.
- Try a harder dataset (e.g., CIFAR-100, a custom Kaggle dataset) to widen the
  performance gap between models.
- Add **Grad-CAM** visualizations to see *where* each network is looking.
- Log experiments with **Weights & Biases** or **TensorBoard** for richer tracking.

---
*Notebook generated for a transfer-learning comparison assignment. Feel free to
tweak `EPOCHS`, `per_class` sample sizes, or unfreeze more layers to push
accuracy further.*
